# Navigation & Visualization Notes  

If you’d like to **jump directly to the visualizations**, use the **navigation bar on the right** of the screen — clicking on a section will take you straight to the below the visualization of the same name.  

> **Note:** Some visualizations may not appear exactly as intended in the **published Kaggle environment** due to limitations in figure size and aspect ratios.  
> - For the **best experience and proper scaling**, I recommend **forking this notebook and running it yourself**.  
> - You can also **collapse the two side panels (left and right)** to maximize the workspace and better interact with the visualizations (e.g., hovering, zooming, or clicking).  
> - Alternatively, you can **download** the exported graphs directly from the **Output** tab:  
>   - Download everything at once as `visualizations.zip`  
>   - Or browse the `visualizations` folder and select only the files you want.  


In [ ]:
!pip install -qqq plotly pandas networkx pyvis matplotlib pillow kaleido squarify
!pip install -qqq --upgrade plotly

In [ ]:
# ============================================================
# Standard library
# ============================================================
import os
import zipfile
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Core scientific stack
# ============================================================
import numpy as np
import pandas as pd
from scipy.interpolate import make_interp_spline
from sklearn.cluster import DBSCAN

# ============================================================
# Plotting libraries
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Matplotlib helpers
# ============================================================
from matplotlib.patches import Circle, Patch
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba, to_hex

# ============================================================
# Graphs / network visualization
# ============================================================
import networkx as nx

# ============================================================
# Image processing
# ============================================================
from PIL import Image, ImageDraw, ImageFont

# ============================================================
# Other math utilities
# ============================================================
from math import pi, cos, sin

# ============================================================
# Plotting style
# ============================================================
plt.style.use('dark_background')
sns.set_palette("husl")


## Cleaning

> **Note:**  
> The dataset has undergone a multi-step collection process, including **scraping, extraction, transformation, and merging**. While careful effort was made to ensure accuracy and consistency, there may still be some **minor errors or inconsistencies** due to the complexity of the process.  
>  
> Additionally, some data is inherently **less precise as originally assigned by DataCamp** and was extracted in that form. For example, a course that primarily covers **PyTorch** may still have its technology listed more broadly as **Python**, which is the case I discovered for one course in my own completed courses.  
>  
> If you notice similar cases in your courses, feel free to add corrections inside the **`clean()`** function located directly below this Markdown cell.  



In [ ]:
def clean():
    # Technology IDs that should not have programming languages
    no_lang_ids = [4,5,7,9,10,11,13,17,18,19,20,21,22,26,29,31,33,34,38,40,43,44]
    
    # Remove programming language for these technologies
    courses_df.loc[courses_df['technology_id'].isin(no_lang_ids), 'programming_language'] = np.nan

    pytorch_keywords = ["pytorch"]
    
    def check_pytorch_keywords(row):
        """Check if pytorch keyword exists in title, short_description, or description"""
        text_fields = [
            str(row['title']).lower(),
            str(row['short_description']).lower() if pd.notna(row['short_description']) else '',
            str(row['description']).lower() if pd.notna(row['description']) else ''
        ]
        combined_text = ' '.join(text_fields)
        return any(keyword in combined_text for keyword in pytorch_keywords)
    
    # Apply PyTorch technology assignment
    pytorch_mask = courses_df.apply(check_pytorch_keywords, axis=1)
    courses_df.loc[pytorch_mask, 'technology_id'] = 27

## Customizing

> **Note:**  
> The **topics, technologies, and programming languages** included in the dataset are based on the **standard categorizations defined by DataCamp**. However, you are free to **customize or extend** them by adding your own categories, tags, or groupings according to what best suits your analysis or learning objectives.  
> To apply such customizations, please write them inside the **`apply_customization`** function located directly below this Markdown cell.  


In [ ]:
def apply_customization():
    """Apply all custom categories: technologies, topics, and keyword-based assignments"""
    global tech_mapping, topic_mapping, technology_colors, topic_colors, merged_df
    
    # add new technology
    keras_row = pd.DataFrame({'technology_id': [45], 'technology_name': ['Keras']})
    tech_mapping = pd.concat([tech_mapping, keras_row], ignore_index=True)
    
    # add new topic
    new_topics = pd.DataFrame({
        'topic_id': [50, 51], 
        'topic_name': ['Deep Learning', 'LLMs']
    })
    topic_mapping = pd.concat([topic_mapping, new_topics], ignore_index=True)
    
    technology_colors['Keras'] = '#D00000'
    topic_colors['Deep Learning'] = '#8B0000'
    topic_colors['LLMs'] = '#4B0082'
    
    
    keras_keywords = ["keras", "tensorflow"]
    
    deep_learning_keywords = [
        "deep learning", "neural network", "neural networks", "cnn", "rnn", "lstm", 
        "gru", "gan", "generative adversarial", "autoencoder", "convolutional neural",
        "recurrent neural", "backpropagation", "hidden layer", "activation function", 
        "dropout", "batch normalization", "deep neural","PyTorch","keras"
    ]

    # More specific LLM keywords
    llm_keywords = [
        "llm", "llms", "large language model", "large language models", "langchain",
        "lang chain", "chatgpt", "gpt", "openai", "hugging face", 
        "generative ai", "generative artificial intelligence", "prompt engineering",
        "prompt engineer", "retrieval augmented generation", "rag",
        "vector database", "vector embedding", "text generation", "language model",
        "fine-tuning", "fine tuning", "pre-trained language", "pretrained language",
        "llm applications", "language modeling", "llmops",
    ]
    
    # Exclusion keywords to prevent wrong assignments
    deep_learning_exclusions = [
        "joining data", "pandas", "dataframe", "data manipulation", "sql join",
        "merge", "concat", "combine datasets", "understanding artificial intelligence",
        "understanding data engineering", "llmops"
    ]
    
    llm_exclusions = [
        "statistics", "exploratory data", "github", "preprocessing", "object-oriented",
        "reinforcement learning", "time series", "data engineering"
    ]
    
    # Helpers
    def check_keywords(row, keywords, exclusions=None):
        """Check if any keyword exists but no exclusion terms exist"""
        text_fields = [
            str(row['title']).lower(),
            str(row['short_description']).lower() if pd.notna(row['short_description']) else '',
            str(row['description']).lower() if pd.notna(row['description']) else ''
        ]
        combined_text = ' '.join(text_fields)
        
        # Check for exclusions first
        if exclusions:
            if any(exclusion in combined_text for exclusion in exclusions):
                return False
        
        # Then check for keywords
        return any(keyword in combined_text for keyword in keywords)
    
    # Re-merge
    merged_df = courses_df.copy()
    merged_df = merged_df.merge(topic_mapping, on='topic_id', how='left')
    merged_df = merged_df.merge(tech_mapping, on='technology_id', how='left')
    merged_df['topic_name'] = merged_df['topic_name'].fillna('Uncategorized')
    merged_df['technology_name'] = merged_df['technology_name'].fillna('General')
    merged_df['programming_language'] = merged_df['programming_language'].fillna('general')
    
    
    # Assign Keras technology (technology_id = 45)
    keras_mask = merged_df.apply(lambda row: check_keywords(row, keras_keywords), axis=1)
    merged_df.loc[keras_mask, 'technology_id'] = 45
    merged_df.loc[keras_mask, 'technology_name'] = 'Keras'
    
    # Assign Deep Learning topic (topic_id = 50)
    # Include courses with PyTorch or Keras technology + keyword matches (with exclusions)
    dl_mask = (
        merged_df.apply(lambda row: check_keywords(row, deep_learning_keywords, deep_learning_exclusions), axis=1) |
        (merged_df['technology_name'] == 'PyTorch') |
        (merged_df['technology_name'] == 'Keras')
    )
    merged_df.loc[dl_mask, 'topic_id'] = 50
    merged_df.loc[dl_mask, 'topic_name'] = 'Deep Learning'
    
    # Assign LLMs topic (topic_id = 51) - after Deep Learning assignment
    # Exclude courses already assigned to Deep Learning and apply exclusions
    llm_mask = (
        merged_df.apply(lambda row: check_keywords(row, llm_keywords, llm_exclusions), axis=1) &
        (merged_df['topic_name'] != 'Deep Learning')  # Exclude Deep Learning courses
    )
    merged_df.loc[llm_mask, 'topic_id'] = 51
    merged_df.loc[llm_mask, 'topic_name'] = 'LLMs'
    
    print("🎯 Customization complete!")

# Visualizations:

In [ ]:
# color palettes and global dataframe variables
language_colors = {
    'python': '#3776ab',
    'r': '#276dc3',
    'sql': '#f29111',
    'scala': '#dc322f',
    'shell': '#4eaa25',
    'spreadsheets': '#34a853', 
    'general': '#666666'  
}

technology_colors = {
    'R': '#276dc3',
    'Python': '#3776ab',
    'SQL': '#f29111',
    'Git': '#F05032',
    'Shell': '#4eaa25',
    'Google Sheets': '#34A853',
    'Theory': '#8B4789',
    'Scala': '#DC322F',
    'Tableau': '#E97627',
    'Excel': '#1D6F42',
    'Power BI': '#F2C811',
    'Julia': '#9558B2',
    'Docker': '#2496ED',
    'Snowflake': '#29B5E8',
    'Redshift': '#E31E2D',
    'BigQuery': '#4285F4',
    'Airflow': '#017CEE',
    'AWS': '#FF9900',
    'Databricks': '#FF3621',
    'dbt': '#FF694B',
    'MLflow': '#0194E2',
    'OpenAI': '#10A37F',
    'ChatGPT': '#74AA9C',
    'PyTorch': '#EE4C2C',
    'Spark': '#E25A1C',
    'Azure': '#0078D4',
    'Kubernetes': '#326CE5',
    'DVC': '#945DD5',
    'Kafka': '#231F20',
    'Alteryx': '#0078C0',
    'Java': '#007396',
    'Llama': '#FF6B35',
    'KNIME': '#FDD900',
    'FastAPI': '#009688',
    'Microsoft Copilot': '#0066CC',
    'DataLab': '#4B8BBE',
    'Sigma': '#5936D8',
    'General': '#666666'  
}

difficulty_colors = {
    1: '#90EE90',  # Light Green - Beginner
    2: '#FFD700',  # Gold - Intermediate
    3: '#FF6B6B'   # Coral Red - Advanced
}

topic_colors = {
    'Data Manipulation': '#FF9F40',
    'Data Visualization': '#36C5F0',
    'Reporting': '#FFC107',
    'Machine Learning': '#E01E5A',
    'Probability & Statistics': '#9C27B0',
    'Importing & Cleaning Data': '#00BCD4',
    'Applied Finance': '#4CAF50',
    'Programming': '#2EB67D',
    'Data Management': '#3F51B5',
    'Data Engineering': '#00A8E8',
    'Artificial Intelligence': '#6A0DAD',
    'Data Literacy': '#FF6B6B',
    'Data Preparation': '#795548',
    'Exploratory Data Analysis': '#FF5722',
    'Data Warehouse': '#607D8B',
    'Leadership': '#E91E63',
    'Cloud': '#03A9F4',
    'Uncategorized': '#95a5a6'  
}

# Global dataframes (populated by load_data())
courses_df = None
topic_mapping = None
tech_mapping = None
tracks_df = None
merged_df = None
completed_tracks = None


In [ ]:
# helper to identify completed tracks
def identify_completed_tracks():
    """Identify which tracks are completed (all courses in track exist in courses.csv)"""
    global courses_df, tracks_df, completed_tracks
    completed_tracks_list = []
    
    for _, track in tracks_df.iterrows():
        # Get course titles from the track
        track_courses = [course.strip() for course in track['course_titles'].split(',')]
        
        # Check if all courses in this track exist in our completed courses
        completed_courses_set = set(courses_df['title'])
        track_courses_set = set(track_courses)
        
        # If all track courses are in completed courses, track is completed
        if track_courses_set.issubset(completed_courses_set):
            completed_tracks_list.append(track)
    
    completed_tracks = pd.DataFrame(completed_tracks_list)


In [ ]:
# load CSVs into globals
def load_data():
    """Load all CSV files"""
    global courses_df, topic_mapping, tech_mapping, tracks_df, merged_df, completed_tracks
    try:
        courses_df = pd.read_csv('/kaggle/input/datacamp-courses-metadata/courses.csv')
        topic_mapping = pd.read_csv('/kaggle/input/datacamp-courses-metadata/topic_mapping.csv')
        tech_mapping = pd.read_csv('/kaggle/input/datacamp-courses-metadata/technology_mapping.csv')
        tracks_df = pd.read_csv('/kaggle/input/datacamp-courses-metadata/all_tracks.csv')

        # Clean the data
        clean()

        # Merge the data
        merged_df = courses_df.copy()
        merged_df = merged_df.merge(topic_mapping, on='topic_id', how='left')
        merged_df = merged_df.merge(tech_mapping, on='technology_id', how='left')

        # Fill missing values
        merged_df['topic_name'] = merged_df['topic_name'].fillna('Uncategorized')
        merged_df['technology_name'] = merged_df['technology_name'].fillna('General')
        merged_df['programming_language'] = merged_df['programming_language'].fillna('general')

        # Apply customizations
        apply_customization()

        # Identify completed tracks
        identify_completed_tracks()

        print(f"✅ Data loaded successfully!")
        print(f"📚 Total courses: {len(merged_df)}")
        print(f"🏆 Total XP: {merged_df['xp'].sum():,}")
        print(f"⏱️ Total hours: {merged_df['time_needed_in_hours'].sum():.1f}")
        print(f"🎯 Completed tracks: {len(completed_tracks)}")

    except FileNotFoundError as e:
        print(f"❌ Error loading files: {e}")
        print("Please ensure all CSV files are in the current directory")



load_data()


In [ ]:
# Radial knowledge tree 
def create_radial_knowledge_tree():
    if merged_df is None:
        print("Please load data first!")
        return
        
    # Group by topic and technology
    topic_stats = merged_df.groupby('topic_name').agg({
        'xp': 'sum',
        'time_needed_in_hours': 'sum',
        'id': 'count',
        'difficulty_level': 'mean'
    }).reset_index()
    topic_stats.columns = ['topic_name', 'total_xp', 'total_hours', 'course_count', 'avg_difficulty']
    
    # Create the radial tree visualization
    fig = go.Figure()
    
    # starfield background
    np.random.seed(42)
    n_stars = 150
    star_angles = np.random.uniform(0, 2 * np.pi, n_stars)
    star_radii = np.random.uniform(0.5, 5.5, n_stars)
    star_x = star_radii * np.cos(star_angles)
    star_y = star_radii * np.sin(star_angles)

    star_sizes = np.random.choice([1.5, 2.5, 4.0], size=n_stars, p=[0.6, 0.3, 0.1])
    star_opacities = np.random.choice([0.2, 0.4, 0.7], size=n_stars, p=[0.5, 0.35, 0.15])

    fig.add_trace(go.Scatter(
        x=star_x, y=star_y,
        mode='markers',
        marker=dict(size=star_sizes, color='white', opacity=star_opacities, symbol='circle'),
        hoverinfo='skip',
        showlegend=False,
        name='Stars'
    ))
    
    # Soft nebula background
    num_nebula = 5
    nebula_x = [np.random.uniform(0, 6), np.random.uniform(-6, 0),
                np.random.uniform(0, 6), np.random.uniform(-6, 0),
                np.random.uniform(4, 6)]  
    nebula_y = [np.random.uniform(0, 6), np.random.uniform(3, 6),
                np.random.uniform(-6, 0), np.random.uniform(-6, 0),
                np.random.uniform(4, 6)]  
    nebula_colors = [
        'rgba(100,149,237,0.08)',
        'rgba(147,112,219,0.08)',
        'rgba(255,182,193,0.06)',
        'rgba(144,238,144,0.06)',
        'rgba(255,105,180,0.07)'  
    ]
    for i in range(num_nebula):
        fig.add_trace(go.Scatter(
            x=[nebula_x[i]],
            y=[nebula_y[i]],
            mode='markers',
            marker=dict(
                size=np.random.uniform(80, 100),
                color=nebula_colors[i],
                symbol='circle',
                line=dict(width=0)
            ),
            hoverinfo='skip',
            showlegend=False,
            name='Nebula'
        ))

    # Calculate optimal orbital radii based on number of topics
    num_topics = len(topic_stats)
    
    # Distribute topics across different orbital rings
    if num_topics <= 4:
        orbit_radii = [3]
    elif num_topics <= 8:
        orbit_radii = [2.5, 4]
    else:
        orbit_radii = [2, 3.5, 5]
    
    # Draw orbital rings only where topics will be placed
    theta = np.linspace(0, 2*np.pi, 100)
    for radius in orbit_radii:
        orbit_x = radius * np.cos(theta)
        orbit_y = radius * np.sin(theta)
        fig.add_trace(go.Scatter(
            x=orbit_x, y=orbit_y,
            mode='lines',
            line=dict(width=1, color='rgba(255,255,255,0.15)', dash='dot'),
            hoverinfo='skip',
            showlegend=False,
            name='Orbit'
        ))
    
    # Add soft orange background circle behind the sun
    fig.add_trace(go.Scatter(
        x=[0],
        y=[0],
        mode='markers',
        marker=dict(
            size=100,  
            color='rgba(255,165,86,0.08)',
            symbol='circle',
            line=dict(width=0)
        ),
        hoverinfo='skip',
        showlegend=False,
        name='Sun Background'
    ))
    
    # Add sun-like glowing effect layers
    glow_layers = [
        {'size': 125, 'opacity': 0.12, 'color': 'rgba(255,165,0,0.8)', 'line_width': 0},   
        {'size': 108, 'opacity': 0.18, 'color': 'rgba(255,140,0,0.9)', 'line_width': 0},   
        {'size': 92, 'opacity': 0.25, 'color': 'rgba(255,215,0,0.95)', 'line_width': 0},   
        {'size': 75, 'opacity': 0.35, 'color': 'rgba(255,255,0,0.9)', 'line_width': 0},    
        {'size': 67, 'opacity': 0.9, 'color': '#FFD700', 'line_width': 3}                   
    ]

    for i, layer in enumerate(glow_layers):
        line_dict = {}
        if layer['line_width'] > 0:
            line_dict = dict(width=layer['line_width'], color='white')
        else:
            line_dict = dict(width=0)
        
        fig.add_trace(go.Scatter(
            x=[0],
            y=[0],
            mode='markers',
            marker=dict(
                size=layer['size'],
                color=layer['color'],
                symbol='circle',
                line=line_dict,
                opacity=layer['opacity']
            ),
            hoverinfo='skip',
            name='Sun Core' if i == len(glow_layers)-1 else 'Sun Glow',
            showlegend=False
        ))
    
    # Center node (Sun)
    total_xp = merged_df["xp"].sum()
    total_courses = len(merged_df)
    
    fig.add_trace(go.Scatter(
        x=[0], y=[0],
        mode='text',
        text=f'<b style="font-family: Arial Black; font-size: 12px; color: #2C2C2C;">{total_xp:,} XP</b><br><b style="font-family: Arial; font-size: 10px; color: #404040;">{total_courses} Courses</b>',  
        textposition="middle center",
        name='Center',
        hoverinfo='skip',
        showlegend=False
    ))

    # Topics placement
    topic_stats_sorted = topic_stats.sort_values('total_xp', ascending=True).reset_index(drop=True)
    topic_x, topic_y, topic_colors_list, topic_sizes, topic_texts = [], [], [], [], []

    # Calculate min and max difficulty for gradient mapping
    min_difficulty = topic_stats_sorted['avg_difficulty'].min()
    max_difficulty = topic_stats_sorted['avg_difficulty'].max()

    topic_idx = 0
    for orbit_idx, radius in enumerate(orbit_radii):
        num_in_orbit = int(np.ceil(num_topics / len(orbit_radii)))
        base_separation = 2 * np.pi / num_in_orbit
        start_offset = (orbit_idx * 0.7) % (2 * np.pi)
        orbit_angles = [(start_offset + i * base_separation) % (2*np.pi) for i in range(num_in_orbit)]
        
        for angle in orbit_angles:
            if topic_idx >= len(topic_stats_sorted): break
            topic = topic_stats_sorted.iloc[topic_idx]
            x, y = radius * cos(angle), radius * sin(angle)
            topic_x.append(x); topic_y.append(y)
            size = 30 + (topic['total_xp'] / max(topic_stats_sorted['total_xp'])) * 40
            topic_sizes.append(size)
            
            # gradient from green -> yellow -> red
            if max_difficulty > min_difficulty:
                # Normalize difficulty to 0-1 range
                normalized_difficulty = (topic['avg_difficulty'] - min_difficulty) / (max_difficulty - min_difficulty)
            else:
                normalized_difficulty = 0

            # Create smooth gradient: Green (0) -> Yellow (0.5) -> Red (1)
            if normalized_difficulty <= 0.5:
                # Green to Yellow transition (0 to 0.5)
                progress = normalized_difficulty * 2  # Scale to 0-1
                red_component = int(50 + progress * 205)    # 50 to 255 (green to yellow)
                green_component = 255                       # Keep green component high
                blue_component = int(50 * (1 - progress))   # 50 to 0
            else:
                # Yellow to Red transition (0.5 to 1)
                progress = (normalized_difficulty - 0.5) * 2  # Scale to 0-1
                red_component = 255                         # Keep red high
                green_component = int(255 * (1 - progress)) # 255 to 0 (yellow to red)
                blue_component = 0                          # Keep blue at 0

            color = f'rgb({red_component}, {green_component}, {blue_component})'
            
            topic_colors_list.append(color)
            topic_texts.append(f"{topic['topic_name']}<br>{topic['course_count']} courses")
            fig.add_trace(go.Scatter(x=[0, x], y=[0, y], mode='lines',
                line=dict(width=2, color='rgba(255,255,255,0.3)'),
                hoverinfo='skip', showlegend=False))
            topic_idx += 1

    # Add glowing topic planets
    for x, y, size, color, text, custom in zip(
        topic_x, topic_y, topic_sizes, topic_colors_list, topic_texts,
        topic_stats_sorted[['total_xp', 'total_hours', 'avg_difficulty']].values
    ):
        # Glow layers
        for k, alpha in [(2.2, 0.06), (1.6, 0.1)]:
            fig.add_trace(go.Scatter(
                x=[x], y=[y], mode='markers',
                marker=dict(size=size*k, color=color, opacity=alpha, symbol='circle', line=dict(width=0)),
                hoverinfo='skip', showlegend=False
            ))
        # Main planet
        topic_name = text.split('<br>')[0]
        course_count = text.split('<br>')[1]
        hover_text = f"<b>{topic_name}</b><br><b>{course_count}</b><br>Total XP: {custom[0]:,}<br>Total Hours: {custom[1]:.1f}<br>Avg Difficulty: {custom[2]:.1f}"
        fig.add_trace(go.Scatter(
            x=[x], y=[y], mode='markers+text',
            marker=dict(size=size, color=color, line=dict(width=2, color='white')),
            text=text, textposition="middle center",
            textfont=dict(size=9, color='#525252', family='Arial Black'),
            name='Topics',
            hovertext=hover_text,
            hoverinfo='text',
            showlegend=False
        ))

    # Add courses around their topics
    for topic_idx, topic_name in enumerate(topic_stats_sorted['topic_name']):
        if topic_idx >= len(topic_x):
            break
            
        topic_courses = merged_df[merged_df['topic_name'] == topic_name]
        
        # Arrange courses around their topic
        num_courses = len(topic_courses)
        if num_courses > 1:
            course_angles = np.linspace(0, 2*pi, num_courses, endpoint=False)
        else:
            course_angles = [0]
        
        course_x, course_y = [], []
        course_colors = []
        course_sizes = []
        course_texts = []
        
        for i, (_, course) in enumerate(topic_courses.iterrows()):
            course_angle = course_angles[i]
            course_radius = 0.9
            
            base_x, base_y = topic_x[topic_idx], topic_y[topic_idx]
            x = base_x + course_radius * cos(course_angle)
            y = base_y + course_radius * sin(course_angle)
            
            course_x.append(x)
            course_y.append(y)
            
            # Size based on XP 
            size = 6 + (course['xp'] / max(merged_df['xp'])) * 10
            course_sizes.append(size)
            
            # Color based on technology instead of programming language
            tech = course['technology_name']
            color = technology_colors.get(tech, '#666666')
            course_colors.append(color)
            
            # Truncate title for display
            title = course['title'][:30] + '...' if len(course['title']) > 30 else course['title']
            course_texts.append(title)
            
            # Add connection from topic to course
            fig.add_trace(go.Scatter(
                x=[base_x, x], y=[base_y, y],
                mode='lines',
                line=dict(width=1, color='rgba(255,255,255,0.2)'),
                hoverinfo='skip',
                showlegend=False
            ))
        
        # Add course nodes
        fig.add_trace(go.Scatter(
            x=course_x, y=course_y,
            mode='markers',
            marker=dict(size=course_sizes, color=course_colors,
                    line=dict(width=1, color='white')),
            name=f'{topic_name} Courses',
            hovertemplate='<b>%{customdata[0]}</b><br>' +
                        'XP: %{customdata[1]}<br>' +
                        'Hours: %{customdata[2]}<br>' +
                        'Technology: %{customdata[3]}<br>' +
                        'Difficulty: %{customdata[4]}<extra></extra>',
            customdata=topic_courses[['title', 'xp', 'time_needed_in_hours', 
                                    'technology_name', 'difficulty_level']].values,
            showlegend=False
        ))
    
    # layout with 
    fig.update_layout(
        title=dict(
            text='✨ Knowledge Galaxy ✨<br><sub>Courses connected by topics and technologies</sub>',
            x=0.5,
            font=dict(size=24, color='white')
        ),
        showlegend=False,
        paper_bgcolor='rgba(10,10,20,1)',
        plot_bgcolor='rgba(10,10,20,1)',
        font=dict(color='white'),
        width=1000,
        height=800,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-7, 7]),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-7, 7])
    )
    
    return fig

# Run and display 
radial_fig = create_radial_knowledge_tree()
if radial_fig is not None:
    radial_fig.show()


## Knowledge Galaxy

### How to Interact:
**🖱️ Mouse Controls:**
- **Hover** over planets and courses to see detailed information
- **Click and drag** to pan around the galaxy
- **Scroll/pinch** to zoom in and out, or use the zoom tools (magnifier icons) at the top-right corner
- **Double-click** empty space to reset view

**🎨 Visual Guide:**
- **Planet colors**: Green (beginner) → Yellow (intermediate) → Red (advanced)
- **Planet sizes**: Larger = more total XP in that topic
- **Course satellites**: Small dots around each topic planet

In [ ]:
# Completed tracks sunburst (3-level)
def create_completed_tracks_sunburst():
    """Create a 3-level sunburst chart for completed tracks with difficulty-based coloring (soft averages)."""
    if completed_tracks is None or len(completed_tracks) == 0:
        print("No completed tracks to visualize!")
        return None
        
    # Prepare hierarchical data with exactly 3 levels
    hierarchical_data = []

    # Dictionary to hold difficulty values for each node
    difficulty_map = {}

    # Level 1: Track Type (Career vs Skill) - Inner circle
    career_tracks = completed_tracks[completed_tracks['is_career'] == 'Yes']
    skill_tracks = completed_tracks[completed_tracks['is_career'] == 'No']

    if len(career_tracks) > 0:
        hierarchical_data.append({
            'ids': 'Career Track',
            'labels': 'Career Track',
            'parents': '',
            'values': career_tracks['total_xp'].sum()
        })

    if len(skill_tracks) > 0:
        hierarchical_data.append({
            'ids': 'Skill Track', 
            'labels': 'Skill Track',
            'parents': '',
            'values': skill_tracks['total_xp'].sum()
        })

    # Level 2 and 3: Tracks and Courses
    for _, track in completed_tracks.iterrows():
        track_type = 'Career Track' if track['is_career'] == 'Yes' else 'Skill Track'
        
        # Track node
        hierarchical_data.append({
            'ids': track['track_title'],
            'labels': track['track_title'],
            'parents': track_type,
            'values': track['total_xp']
        })
        
        # Gather course difficulties for averaging
        course_difficulties = []
        track_courses = [course.strip() for course in track['course_titles'].split(',')]
        for course_title in track_courses:
            course_data = merged_df[merged_df['title'] == course_title]
            if len(course_data) > 0:
                course = course_data.iloc[0]
                course_id = f"{course_title}_{track['track_title']}"
                hierarchical_data.append({
                    'ids': course_id,
                    'labels': course_title,
                    'parents': track['track_title'],
                    'values': course['xp']
                })
                # Store course difficulty
                difficulty_map[course_id] = course['difficulty_level']
                course_difficulties.append(course['difficulty_level'])
        
        # Track difficulty = average of its courses
        if course_difficulties:
            difficulty_map[track['track_title']] = np.mean(course_difficulties)

    # Assign parent difficulties (Career / Skill) = average of their tracks
    for parent in ['Career Track', 'Skill Track']:
        child_tracks = completed_tracks[completed_tracks['is_career'] == ('Yes' if parent == 'Career Track' else 'No')]
        track_diffs = [difficulty_map[t] for t in child_tracks['track_title'] if t in difficulty_map]
        if track_diffs:
            difficulty_map[parent] = np.mean(track_diffs)

    # Convert to DataFrame
    hierarchy_df = pd.DataFrame(hierarchical_data)

    # Map difficulties back to nodes
    hierarchy_df['difficulty'] = hierarchy_df['ids'].map(difficulty_map)
    difficulty_values = hierarchy_df['difficulty'].fillna(0).tolist()

    # Create the sunburst with difficulty-based coloring
    fig = go.Figure(go.Sunburst(
        ids=hierarchy_df['ids'],
        labels=hierarchy_df['labels'],
        parents=hierarchy_df['parents'],
        values=hierarchy_df['values'],
        branchvalues="total",
        maxdepth=3,
        hovertemplate='<b>%{label}</b><br>XP: %{value:,}<br>Difficulty: %{color:.2f}<extra></extra>',
        marker=dict(
            colors=difficulty_values,
            colorscale='Turbo',
            cmin=1, cmax=3,  
            showscale=True,
            colorbar=dict(
                title=dict(
                    text="Difficulty Level",
                    font=dict(color='white')
                ),
                tickfont=dict(color='white')
            ),
            line=dict(color="#FFFFFF", width=2)
        )
    ))

    fig.update_layout(
        title=dict(
            text='🌟 Completed Tracks Sunburst 🌟<br><sub>3-Level Hierarchy: Track Type → Tracks → Courses</sub>',
            x=0.5,
            font=dict(size=24, color='white')
        ),
        paper_bgcolor='rgba(10,10,30,1)',
        plot_bgcolor='rgba(10,10,30,1)',
        font=dict(color='white', size=12),
        width=900,
        height=800,
        margin=dict(t=100, l=50, r=50, b=50)
    )
    
    return fig

# Run and display 
sunburst_fig = create_completed_tracks_sunburst()
if sunburst_fig is not None:
    sunburst_fig.show()


## Tracks Sunburst

### How to Interact

**🖱️ Mouse Controls:**
- **Click** on any segment to zoom in and focus on that branch
- **Click center** to zoom back out to previous level
- **Hover** over segments to see XP values and difficulty levels

**🎨 Visual Guide:**
- **Inner ring**: Track types (Career vs Skill tracks)
- **Middle ring**: Individual track names
- **Outer ring**: Individual courses within each track
- **Colors**: Blue (easy) → Green → Yellow → Red (difficult) based on average difficulty

In [ ]:
# Achievement dashboard 
def create_achievement_dashboard():
    if merged_df is None:
        print("Please load data first!")
        return
        
    # Calculate statistics
    total_topics = merged_df['topic_name'].nunique()
    total_hours = merged_df['time_needed_in_hours'].sum()
    total_courses = len(merged_df)
    total_completed_tracks = len(completed_tracks) if completed_tracks is not None else 0
    total_technologies = merged_df['technology_name'].nunique() 
    avg_difficulty = merged_df['difficulty_level'].mean()
    
    # Create subplots 
    fig = make_subplots(
        rows=3, cols=4,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"type": "bar", "colspan": 2}, None, {"type": "scatter"}, {"type": "pie"}],
            [{"type": "bar", "colspan": 4}, None, None, None]
        ],
        subplot_titles=[
            "", "", "", "",
            "Top Technologies", "Technology Constellation", "Difficulty Distribution",
            "XP Distribution by Topic"
        ],
        vertical_spacing=0.08,
        horizontal_spacing=0.03
    )
    
    # KPI Indicators
    fig.add_trace(go.Indicator(
        mode="number+delta",
        value=total_topics,
        number={'font': {'size': 32, 'color': 'gold'}},
        title={"text": "🎯 Topics Mastered", "font": {"size": 18}}, 
    ), row=1, col=1)
    
    fig.add_trace(go.Indicator(
        mode="number+delta",
        value=total_hours,
        number={'font': {'size': 32, 'color': 'lightblue'}, 'suffix': 'h'},
        title={"text": "⏱️ Learning Time", "font": {"size": 18}}, 
    ), row=1, col=2)
    
    fig.add_trace(go.Indicator(
        mode="number+delta",
        value=total_courses,
        number={'font': {'size': 32, 'color': 'lightgreen'}},
        title={"text": "📚 Courses Completed", "font": {"size": 18}},
    ), row=1, col=3)
    
    fig.add_trace(go.Indicator(
        mode="number+delta",
        value=total_completed_tracks,
        number={'font': {'size': 32, 'color': 'orange'}},
        title={"text": "🏆 Tracks", "font": {"size": 18}},  
    ), row=1, col=4)
    
    # Technologies bar chart
    tech_counts = merged_df['technology_name'].value_counts().head(8)
    tech_counts = tech_counts.sort_values(ascending=True)
    
    colors = []
    for tech in tech_counts.index:
        color = technology_colors.get(tech, '#666666')
        colors.append(color)
    
    fig.add_trace(go.Bar(
        x=tech_counts.values,
        y=tech_counts.index,
        orientation='h',
        marker=dict(
            color=colors,
            line=dict(color='white', width=1),
            pattern_shape="/"
        ),
        name='Technologies',
        hovertemplate='<b>%{y}</b><br>Courses: %{x}<extra></extra>'
    ), row=2, col=1)
    
    # Difficulty distribution pie
    diff_counts = merged_df['difficulty_level'].value_counts().sort_index()
    diff_labels = [f'Level {i}' for i in diff_counts.index]
    
    diff_colors = []
    for i in diff_counts.index:
        color = difficulty_colors.get(i, '#666666')
        diff_colors.append(color)

    fig.add_trace(go.Pie(
        labels=diff_labels,
        values=diff_counts.values,
        marker=dict(
            colors=diff_colors,
            line=dict(color='white', width=2)
        ),
        name='Difficulty',
        hovertemplate='<b>%{label}</b><br>Courses: %{value}<br>Percentage: %{percent}<extra></extra>'
    ), row=2, col=4)
    
    # Technology Constellation Plot
    tech_courses = {}
    for _, course in merged_df.iterrows():
        tech = course['technology_name']
        if tech not in tech_courses:
            tech_courses[tech] = {
                'courses': 0,
                'total_xp': 0,
                'topics': set(),
                'difficulties': []
            }
        tech_courses[tech]['courses'] += 1
        tech_courses[tech]['total_xp'] += course['xp']
        tech_courses[tech]['topics'].add(course['topic_name'])
        tech_courses[tech]['difficulties'].append(course['difficulty_level'])
    
    # Convert to visualization data
    breadth_data = []
    for tech, data in tech_courses.items():
        if data['courses'] >= 1:
            breadth_score = len(data['topics'])
            depth_score = data['total_xp'] / data['courses']
            avg_difficulty = np.mean(data['difficulties'])
            breadth_data.append({
                'technology': tech,
                'breadth': breadth_score,
                'depth': depth_score,
                'courses': data['courses'],
                'avg_difficulty': avg_difficulty
            })
    
    if breadth_data:
        breadth_df = pd.DataFrame(breadth_data)
        
        # Create network-style plot showing learning technologies
        angles = np.linspace(0, 2*np.pi, len(breadth_df), endpoint=False)
        
        tech_x, tech_y = [], []
        tech_colors = []
        tech_texts = []
        
        for i, (_, row) in enumerate(breadth_df.iterrows()):
            radius = 3.0
            x = radius * np.cos(angles[i])
            y = radius * np.sin(angles[i])
            
            tech_x.append(x)
            tech_y.append(y)
            
            if row['avg_difficulty'] <= 1.5:
                color = difficulty_colors[1]
            elif row['avg_difficulty'] <= 2.5:
                color = difficulty_colors[2]
            else:
                color = difficulty_colors[3]
            tech_colors.append(color)
            
            # Technology name
            short_name = row['technology'][:12] + '...' if len(row['technology']) > 12 else row['technology']
            tech_texts.append(short_name)
        
        # Add technology points as hexagons
        fig.add_trace(go.Scatter(
            x=tech_x,
            y=tech_y,
            mode='markers+text',
            marker=dict(
                size=25,
                color=tech_colors,
                symbol='hexagon',
                line=dict(width=3, color='white'),
                opacity=0.9
            ),
            text=tech_texts,
            textposition="top center",
            textfont=dict(size=9, color='white', family='Arial Black'),
            hovertemplate='<b>%{text}</b><br>Courses: %{customdata[0]}<br>Avg Difficulty: %{customdata[1]:.1f}<extra></extra>',
            customdata=[[row['courses'], row['avg_difficulty']] for _, row in breadth_df.iterrows()],
            name='Technologies',
            showlegend=False
        ), row=2, col=3)
        
        # Add connecting lines from center to each technology
        for x, y in zip(tech_x, tech_y):
            fig.add_trace(go.Scatter(
                x=[0, x],
                y=[0, y],
                mode='lines',
                line=dict(width=2, color='rgba(255,255,255,0.2)'),
                hoverinfo='skip',
                showlegend=False
            ), row=2, col=3)
        
        # Add soft orange background circle behind the star
        fig.add_trace(go.Scatter(
            x=[0],
            y=[0],
            mode='markers',
            marker=dict(
                size=35,
                color='rgba(255,165,86,0.1)',
                symbol='circle',
                line=dict(width=0)
            ),
            hoverinfo='skip',
            showlegend=False,
            name='Star Background'
        ), row=2, col=3)
        
        # Add glowing star effect layers (from largest/most transparent to smallest/most opaque)
        glow_layers = [
            {'size': 70, 'opacity': 0.15, 'color': 'gold', 'line_width': 0},      # Largest, most transparent
            {'size': 55, 'opacity': 0.25, 'color': 'gold', 'line_width': 0},      # Medium
            {'size': 40, 'opacity': 0.35, 'color': 'gold', 'line_width': 0},      # Smaller
            {'size': 30, 'opacity': 0.9, 'color': '#f2f8ff', 'line_width': 2}     # Original star 
        ]

        for i, layer in enumerate(glow_layers):
            line_dict = {}
            if layer['line_width'] > 0:
                line_dict = dict(width=layer['line_width'], color='rgba(200,200,200,0.8)')
            else:
                line_dict = dict(width=0)
            
            fig.add_trace(go.Scatter(
                x=[0],
                y=[0],
                mode='markers',
                marker=dict(
                    size=layer['size'],
                    color=layer['color'],
                    symbol='star',
                    line=line_dict,
                    opacity=layer['opacity']
                ),
                hoverinfo='skip',
                name='Center' if i == len(glow_layers)-1 else 'Glow',
                showlegend=False
            ), row=2, col=3)
    
    # Add Technologies annotation above the star in constellation
    fig.add_annotation(
        text=f'{total_technologies} Technologies',
        x=0.63, y=0.55,  # Position above the star in constellation area
        xref="paper", yref="paper",
        showarrow=False,
        font=dict(size=11.5, color='lightblue'),
        bgcolor='rgba(0,0,0,0.6)',
        bordercolor='lightblue',
        borderwidth=1,
        borderpad=3
    )
    
    # XP distribution by topic
    topic_xp_dist = []
    for topic in merged_df['topic_name'].unique():
        topic_courses = merged_df[merged_df['topic_name'] == topic]
        for _, course in topic_courses.iterrows():
            topic_xp_dist.append({
                'topic': topic,
                'course': course['title'][:30] + '...' if len(course['title']) > 30 else course['title'],
                'xp': course['xp'],
                'difficulty': course['difficulty_level'],
                'language': course['programming_language']
            })
    
    topic_xp_df = pd.DataFrame(topic_xp_dist)
    
    marker_colors = []
    for topic in topic_xp_df['topic']:
        color = topic_colors.get(topic, '#95a5a6')
        marker_colors.append(color)
    
    fig.add_trace(go.Bar(
        x=topic_xp_df['topic'],
        y=topic_xp_df['xp'],
        marker_color=marker_colors,
        text=topic_xp_df['course'],
        hovertemplate='<b>%{text}</b><br>Topic: %{x}<br>XP: %{y:,}<extra></extra>',
        name='Course XP by Topic'
    ), row=3, col=1)
    
    # Update subplot axes
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, range=[-4, 4], showline=False, row=2, col=3)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, range=[-4, 4], showline=False, row=2, col=3)
    
    fig.update_layout(
        title=dict(
            text='📊 Achievement Dashboard 📊',
            x=0.5,
            font=dict(size=24, color='white')
        ),
        paper_bgcolor='rgba(10,10,20,1)',
        plot_bgcolor='rgba(10,10,20,1)',
        font=dict(color='white'),
        height=1000,
        showlegend=False
    )
    
    return fig

# Run and display the dashboard
dashboard_fig = create_achievement_dashboard()
if dashboard_fig is not None:
    dashboard_fig.show()


## Achievements dashboard 

### How to Interact

**🖱️ Mouse Controls:**
- **Hover** over any chart element to see detailed information
- **Click and drag** to pan within individual charts
- **Use zoom tools** (magnifier icons) at the top-right corner for manual zoom
- **Double-click** on chart areas to reset zoom

**📈 Dashboard Sections:**
- **Top row**: Key performance indicators (KPIs) - total topics, hours, courses, and tracks
- **Middle left**: Top technologies bar chart - courses per technology
- **Middle center**: Technology constellation - hexagonal network view of your tech stack
- **Middle right**: Difficulty distribution pie chart
- **Bottom**: XP distribution showing all courses grouped by topic

In [ ]:
# Track progression analysis
def analyze_longest_track_progression():
    if merged_df is None or completed_tracks is None or len(completed_tracks) == 0:
        print("Please load data first or no completed tracks found!")
        return
        
    # Find the longest track (by course count)
    longest_track = completed_tracks.loc[completed_tracks['course_count'].idxmax()]
    track_title = longest_track['track_title']
    track_courses = [course.strip() for course in longest_track['course_titles'].split(',')]
    
    # Get course data for this track
    track_course_data = []
    for course_title in track_courses:
        course_data = merged_df[merged_df['title'] == course_title]
        if len(course_data) > 0:
            track_course_data.append(course_data.iloc[0])
    
    if not track_course_data:
        print("No matching course data found for this track!")
        return
        
    track_df = pd.DataFrame(track_course_data)
    
    # Calculate completion percentage
    min_subscriptions = track_df['nb_of_subscriptions'].min()
    track_enrollments = int(longest_track['participant_count'].replace(',', ''))
    completion_percentage = (min_subscriptions / track_enrollments) * 100
    
    # Create the visualization
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 16))
    
    # Plot 1: Participation curve 
    track_df_sorted = track_df.sort_values('nb_of_subscriptions', ascending=False).reset_index(drop=True)
    
    # Create colors based on difficulty
    colors = []
    for diff in track_df_sorted['difficulty_level']:
        if diff <= 1:
            colors.append('#90EE90')  # Light Green - Beginner
        elif diff <= 2:
            colors.append('#FFD700')  # Gold - Intermediate  
        else:
            colors.append('#FF6B6B')  # Coral Red - Advanced
    
    # Plot the participation curve
    x_positions = range(len(track_df_sorted))
    participations = track_df_sorted['nb_of_subscriptions'].values
    
    # Create smooth curve
    from scipy.interpolate import make_interp_spline
    x_smooth = np.linspace(0, len(track_df_sorted)-1, 300)
    spl = make_interp_spline(x_positions, participations, k=2)
    participation_smooth = spl(x_smooth)
    
    # Plot smooth curve and fill
    ax1.fill_between(x_smooth, participation_smooth, alpha=0.3, color='cyan')
    ax1.plot(x_smooth, participation_smooth, color='white', linewidth=3)
    
    # Plot individual course points
    bars = ax1.bar(x_positions, participations, color=colors, alpha=0.8, 
                edgecolor='white', linewidth=2)
    
    # Add course labels (rotated for readability)
    course_labels = [title[:25] + '...' if len(title) > 25 else title 
                    for title in track_df_sorted['title']]
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(course_labels, rotation=45, ha='right')
    
    # Create title with proper positioning using figure coordinates
    from matplotlib.patches import Patch
    
    # Clear any existing titles
    ax1.set_title("")
    
    # Position text elements at specific figure coordinates to avoid overlap
    # Line 1 - Main title (y = 0.95)
    fig.text(0.25, 0.95, "Longest Completed Track: ", fontsize=16, fontweight='bold', 
             color='white', ha='left', transform=fig.transFigure)
    fig.text(0.45, 0.95, f"'{track_title}'", fontsize=16, fontweight='bold', 
             color='cyan', ha='left', transform=fig.transFigure)
    fig.text(0.75, 0.95, f" ({len(track_courses)} courses)", fontsize=16, fontweight='bold', 
             color='white', ha='left', transform=fig.transFigure)
    
    # Line 2 - Completion percentage (y = 0.92)  
    fig.text(0.25, 0.92, "Less than ", fontsize=16, fontweight='bold', 
             color='white', ha='left', transform=fig.transFigure)
    fig.text(0.35, 0.92, f"{completion_percentage:.1f}%", fontsize=20, fontweight='bold', 
             color='red', ha='left', transform=fig.transFigure)
    fig.text(0.45, 0.92, " of people enrolled in this track completed it!", fontsize=16, fontweight='bold', 
             color='white', ha='left', transform=fig.transFigure)
    
    # Line 3 - Subtitle (y = 0.89)
    fig.text(0.5, 0.89, "(Courses sorted by participation - highest to lowest | Beginner: Green | Intermediate: Gold | Advanced: Red)", 
             fontsize=11, fontweight='normal', color='lightgray', ha='center', 
             style='italic', transform=fig.transFigure)
    
    ax1.set_ylabel('Number of Participants', color='white')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Format y-axis to show numbers in millions/thousands
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    
    # Plot 2: COMPLETELY REDESIGNED Prerequisites Flow
    import networkx as nx
    G = nx.DiGraph()
    
    # Create title-to-data mapping for quick lookups
    title_to_data = {course['title']: course for course in track_course_data}
    course_titles = list(title_to_data.keys())
    
    # Add all courses as nodes
    for course in track_course_data:
        G.add_node(course['title'], 
                difficulty=course['difficulty_level'],
                xp=course['xp'],
                participants=course['nb_of_subscriptions'])
    
    # prerequisite extraction and validation
    edges_added = 0
    dependency_map = {}
    
    for course in track_course_data:
        current_title = course['title']
        prereq_text = course['prerequisites_titles']
        
        if pd.notna(prereq_text) and str(prereq_text).strip():
            # Clean the prerequisites text
            prereq_text = str(prereq_text).strip()
            
            # Split by multiple delimiters (semicolon, comma, pipe, newline)
            import re
            prereq_candidates = re.split(r'[;,|\n]+', prereq_text)
            
            valid_prereqs = []
            for prereq_raw in prereq_candidates:
                prereq_clean = prereq_raw.strip()
                if len(prereq_clean) < 3:  # Skip very short strings
                    continue
                    
                # Try exact match first
                if prereq_clean in course_titles:
                    valid_prereqs.append(prereq_clean)
                else:
                    # Try fuzzy matching with similarity threshold
                    best_match = None
                    best_score = 0
                    
                    for course_title in course_titles:
                        # Calculate similarity (simple word overlap approach)
                        prereq_words = set(prereq_clean.lower().split())
                        title_words = set(course_title.lower().split())
                        
                        if len(prereq_words) > 0 and len(title_words) > 0:
                            overlap = len(prereq_words.intersection(title_words))
                            similarity = overlap / max(len(prereq_words), len(title_words))
                            
                            # Also check if one contains the other (for partial matches)
                            if (prereq_clean.lower() in course_title.lower() or 
                                course_title.lower() in prereq_clean.lower()):
                                similarity = max(similarity, 0.7)
                        
                            if similarity > best_score and similarity > 0.5:  # Threshold for matching
                                best_score = similarity
                                best_match = course_title
                    
                    if best_match and best_score > 0.5:
                        valid_prereqs.append(best_match)
            
            # Add valid prerequisites as edges
            if valid_prereqs:
                dependency_map[current_title] = valid_prereqs
                for prereq in valid_prereqs:
                    G.add_edge(prereq, current_title)
                    edges_added += 1
    
    # If no prerequisites found, create a logical progression
    if edges_added == 0:
        # Sort courses by difficulty first, then by XP to create logical flow
        sorted_courses = sorted(track_course_data, 
                            key=lambda x: (x['difficulty_level'], x['xp']))
        
        # Create a chain of dependencies based on logical progression
        for i in range(len(sorted_courses) - 1):
            current_course = sorted_courses[i]['title']
            next_course = sorted_courses[i + 1]['title']
            G.add_edge(current_course, next_course)
            edges_added += 1
    
    # Set up the subplot
    ax2.clear()
    ax2.set_facecolor('#0a0a14')  # Darker background
    
    # hierarchical positioning
    try:
        # Try to use graphviz for hierarchical layout (left-to-right flow)
        pos = nx.nx_agraph.graphviz_layout(G, prog='dot', args='-Grankdir=LR -Gsplines=ortho')
    except:
        try:
            # Alternative: create manual hierarchical layout
            levels = {}
            
            # Identify levels based on number of prerequisites
            def get_level(node, visited=None):
                if visited is None:
                    visited = set()
                if node in visited:
                    return 0  # Avoid cycles
                visited.add(node)
                
                predecessors = list(G.predecessors(node))
                if not predecessors:
                    return 0
                else:
                    return 1 + max(get_level(pred, visited.copy()) for pred in predecessors)
            
            # Assign levels to all nodes
            for node in G.nodes():
                levels[node] = get_level(node)
            
            # Create positions based on levels
            level_nodes = {}
            for node, level in levels.items():
                if level not in level_nodes:
                    level_nodes[level] = []
                level_nodes[level].append(node)
            
            pos = {}
            max_level = max(levels.values()) if levels else 0
            
            for level, nodes in level_nodes.items():
                x = level * 6  
                for i, node in enumerate(nodes):
                    y = (i - len(nodes)/2) * 2.5 
                    pos[node] = (x, y)
                    
        except:
            pos = nx.spring_layout(G, k=8, iterations=200, seed=42, scale=5) 
    
    # Add some spacing between nodes to avoid overlaps
    if len(pos) > 1:
        # Calculate minimum distances and add spacing if needed
        positions = list(pos.values())
        min_dist = 1.0  # Minimum desired distance
        
        for node1 in pos:
            for node2 in pos:
                if node1 != node2:
                    dx = pos[node2][0] - pos[node1][0]
                    dy = pos[node2][1] - pos[node1][1]
                    dist = (dx**2 + dy**2)**0.5
                    
                    if dist < min_dist and dist > 0:
                        # Push nodes apart
                        push_factor = (min_dist - dist) / (2 * dist)
                        pos[node1] = (pos[node1][0] - dx * push_factor, 
                                    pos[node1][1] - dy * push_factor)
                        pos[node2] = (pos[node2][0] + dx * push_factor, 
                                    pos[node2][1] + dy * push_factor)
    # edges
    if G.number_of_edges() > 0:
        for edge in G.edges():
            start_pos = pos[edge[0]]
            end_pos = pos[edge[1]]
            
            # Draw straight arrows 
            ax2.annotate('', xy=end_pos, xytext=start_pos,
                        arrowprops=dict(
                            arrowstyle='->', 
                            color='lightsteelblue', 
                            lw=2.5,
                            alpha=0.8,
                            shrinkA=30,  # space around nodes
                            shrinkB=30,
                            mutation_scale=20  # arrow heads
                        ))
    
    # Draw nodes 
    max_xp = max(course['xp'] for course in track_course_data)
    
    for node in G.nodes():
        course_data = title_to_data[node]
        x, y = pos[node]
        
        # Difficulty-based styling
        difficulty = course_data['difficulty_level']
        if difficulty <= 1:
            color = '#90EE90'
            difficulty_label = 'Beginner'
            border_color = '#228B22'
        elif difficulty <= 2:
            color = '#FFD700'
            difficulty_label = 'Intermediate'
            border_color = '#DAA520'
        else:
            color = '#FF6B6B'
            difficulty_label = 'Advanced'
            border_color = '#DC143C'
        
        # Size based on XP 
        size_factor = course_data['xp'] / max_xp
        node_size = 1200 + (size_factor * 800)  
        
        # Draw node 
        circle = ax2.scatter(x, y, s=node_size, c=color, alpha=0.9, 
                        edgecolors=border_color, linewidths=3, zorder=3)
        
        # Course title 
        title_text = node
        if len(title_text) > 20:
            # truncation at word boundaries
            words = title_text.split()
            if len(words) > 3:
                title_text = ' '.join(words[:3]) + '...'
            else:
                title_text = title_text[:20] + '...'
        
        # Main title
        ax2.text(x, y, title_text, ha='center', va='center',
                fontweight='bold', fontsize=9, color='black', zorder=5,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, 
                        alpha=0.95, edgecolor=border_color, linewidth=2))
        
        # XP info below 
        ax2.text(x, y - 0.4, f"{course_data['xp']} XP", 
                ha='center', va='top', fontsize=8, color='white', 
                style='italic', zorder=5)
    
    # title
    ax2.set_title(f"Learning Path Flow for '{track_title}'\n"
                f"Dependencies: {edges_added} | Node size ∝ XP | Color = Difficulty", 
                fontsize=14, fontweight='bold', color='white', pad=20)
    
    # Remove axes and set proper limits
    ax2.axis('equal')
    ax2.axis('off')
    
    # Set limits with proper padding
    if pos:
        x_coords = [pos[node][0] for node in pos]
        y_coords = [pos[node][1] for node in pos]
        padding = 1.0
        ax2.set_xlim(min(x_coords) - padding, max(x_coords) + padding)
        ax2.set_ylim(min(y_coords) - padding, max(y_coords) + padding)
    
    legend_elements = [
        Patch(facecolor='#90EE90', edgecolor='#228B22', linewidth=2, label='Beginner'),
        Patch(facecolor='#FFD700', edgecolor='#DAA520', linewidth=2, label='Intermediate'),
        Patch(facecolor='#FF6B6B', edgecolor='#DC143C', linewidth=2, label='Advanced')
    ]
    
    ax2.legend(handles=legend_elements, loc='upper right', 
            title='Difficulty Levels', title_fontsize=11, fontsize=10,
            facecolor='black', edgecolor='white', framealpha=0.9)
    
    plt.tight_layout()
    return fig

# Run and display 
timeline_fig = analyze_longest_track_progression()
plt.show()

## Longest track progression analysis


> **Note on Interpretation:**  
> The **exact number of people who completed a course** cannot be determined from the publicly available DataCamp information.  
>  
> To provide an **estimate**, we calculated a conservative upper bound:  
> - We took the **minimum number of course subscriptions within a track** (i.e., the course in the track with the least subscriptions).  
> - Then we **divided this value by the total number of people enrolled in the track**.  
>  
> This ensures that the reported completion figures are not overstated, since the actual number of track completions is **guaranteed to be less than or equal to this estimate**.  


In [ ]:
# Personal Knowledge Graph (PKG)
def create_PKG(use_original=False):
    if merged_df is None:
        print("Please load data first!")
        return

    # Choose which data to use
    if use_original:
        # Create original merged data: use the standard topics and technologies assigned by datacamp
        data_to_use = courses_df.copy()
        data_to_use = data_to_use.merge(
            pd.read_csv('/kaggle/input/datacamp-courses-metadata/topic_mapping.csv'), 
            on='topic_id', how='left'
        )
        data_to_use = data_to_use.merge(
            pd.read_csv('/kaggle/input/datacamp-courses-metadata/technology_mapping.csv'), 
            on='technology_id', how='left'
        )
        data_to_use['topic_name'] = data_to_use['topic_name'].fillna('Uncategorized')
        data_to_use['technology_name'] = data_to_use['technology_name'].fillna('General')
        print("📊 Using original categories")
    else:
        data_to_use = merged_df
        print("📊 Using customized categories")

    # Create a selective network
    G = nx.Graph()
    
    # Add nodes 
    for _, course in data_to_use.iterrows():
        G.add_node(course['id'], 
                title=course['title'],
                language=course['programming_language'],
                technology=course['technology_name'],
                topic=course['topic_name'],
                difficulty=course['difficulty_level'],
                xp=course['xp'],
                hours=course['time_needed_in_hours'])
    
    # Create stronge, more meaningful connections
    courses_list = data_to_use.to_dict('records')
    edges_added = 0
    
    # Only connect courses that share technology AND topic (strong relation)
    for i, course1 in enumerate(courses_list):
        for j, course2 in enumerate(courses_list[i+1:], i+1):
            if (course1['technology_name'] == course2['technology_name'] and
                course1['topic_name'] == course2['topic_name']):
                
                # Weight based on similarity
                weight = 2 + abs(course1['difficulty_level'] - course2['difficulty_level'])
                G.add_edge(course1['id'], course2['id'], weight=weight)
                edges_added += 1
    
    print(f"✅ Created network with {G.number_of_nodes()} nodes and {edges_added} edges")
    
    # Use a force-directed layout with clustering
    pos = nx.spring_layout(G, k=3, iterations=100, seed=42)
    
    # Apply DBSCAN clustering for better grouping
    positions = np.array(list(pos.values()))
    if len(positions) > 1:
        clustering = DBSCAN(eps=0.3, min_samples=2).fit(positions)
        labels = clustering.labels_
    else:
        labels = np.zeros(len(positions))
    
    # Create the visualization
    plt.figure(figsize=(16, 12))
    
    # Draw edges with transparency
    if G.number_of_edges() > 0:
        edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
        max_weight = max(edge_weights) if edge_weights else 1
        edge_widths = [0.5 + (w/max_weight) * 2 for w in edge_weights]
        
        nx.draw_networkx_edges(G, pos, alpha=0.3, width=edge_widths, 
                            edge_color='lightblue')
    
    #node drawing
    unique_labels = set(labels)
    colorsc = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colorsc):
        if label == -1:  # Noise points - draw separately with special styling
            noise_nodes = [node for i, node in enumerate(G.nodes()) if labels[i] == label]
            if noise_nodes:
                # Style noise points as stars with gradient colors
                noise_colors = [difficulty_colors.get(G.nodes[node]['difficulty'], '#666666') for node in noise_nodes]
                noise_sizes = [30 + (G.nodes[node]['xp']/max([G.nodes[n]['xp'] for n in G.nodes()])) * 45 for node in noise_nodes] 
                
                nx.draw_networkx_nodes(G, pos, nodelist=noise_nodes,
                                    node_color=noise_colors, node_size=noise_sizes,
                                    alpha=0.9, edgecolors='white', linewidths=2.5,
                                    node_shape='^')  # Triangle shape for noise points
            continue
            
        node_list = [node for i, node in enumerate(G.nodes()) if labels[i] == label]
        
        # node styling 
        e_colors = []
        e_sizes = []
        
        for node in node_list:
            node_data = G.nodes[node]
            
            # Multi-layered color scheme based on difficulty and technology
            base_color = difficulty_colors.get(node_data['difficulty'], '#666666')
            tech_color = technology_colors.get(node_data['technology'], '#888888')
            
            # Blend colors for unique appearance
            base_rgba = to_rgba(base_color)
            tech_rgba = to_rgba(tech_color)
            blended_color = tuple((base_rgba[i] + tech_rgba[i]) / 2 for i in range(3)) + (0.9,)
            e_colors.append(blended_color)
            
            # Dynamic size based on multiple factors (reduced)
            xp_factor = node_data['xp'] / max([G.nodes[n]['xp'] for n in G.nodes()])
            degree_factor = G.degree(node) / max(dict(G.degree()).values()) if G.degree(node) > 0 else 0.1
            size = 40 + (xp_factor * 80) + (degree_factor * 40)
            e_sizes.append(size)
        
        # Draw main cluster nodes  
        scatter = nx.draw_networkx_nodes(G, pos, nodelist=node_list,
                                    node_color=e_colors, node_size=e_sizes,
                                    alpha=0.85, edgecolors='white', linewidths=2.0)
        
        # Add subtle glow effect for important nodes
        important_nodes = [node for node in node_list if G.degree(node) > 2 or G.nodes[node]['xp'] > 3500]
        if important_nodes:
            glow_sizes = [s * 1.3 for node, s in zip(node_list, e_sizes) if node in important_nodes] 
            glow_colors = ['gold' if G.nodes[node]['xp'] > 4500 else 'lightblue' for node in important_nodes]
            
            nx.draw_networkx_nodes(G, pos, nodelist=important_nodes,
                                node_color=glow_colors, node_size=glow_sizes,
                                alpha=0.2, edgecolors='none')
    
    # labeling with positioning
    labels_to_show = {}
    for node in G.nodes():
        title = G.nodes[node]['title']
        # truncation
        words = title.split()
        if len(words) > 0:
            short_title = f"{words[0]}..."
        else:
            short_title = title[:8] + "..." if len(title) > 8 else title
        labels_to_show[node] = short_title
    
    # Draw labels 
    for node, label in labels_to_show.items():
        x, y = pos[node]
        # Add background box for better readability 
        plt.text(x, y, label, ha='center', va='center',
                fontsize=6, fontweight='bold', color='white', alpha=0.8,  
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.8, edgecolor='white'), 
                zorder=10)
    
    # legend 
    tech_counts = {}
    for node in G.nodes():
        tech = G.nodes[node]['technology']
        tech_counts[tech] = tech_counts.get(tech, 0) + 1
    
    top_techs = sorted(tech_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    legend_elements = []
    
    for tech, count in top_techs:
        color = technology_colors.get(tech, '#666666')
        legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                        markerfacecolor=color, markersize=10, 
                                        markeredgecolor='white', markeredgewidth=1,
                                        label=f'{tech} ({count})'))
    
    # Add difficulty level indicators to legend
    legend_elements.append(plt.Line2D([0], [0], linestyle='', label=''))  # Spacer
    for diff, color in difficulty_colors.items():
        level_name = {1: 'Beginner', 2: 'Intermediate', 3: 'Advanced', 4: 'Expert'}.get(diff, f'Level {diff}')
        legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                        markerfacecolor=color, markersize=8,
                                        markeredgecolor='white', markeredgewidth=1,
                                        label=level_name))
    
 #   plt.legend(handles=legend_elements, title='Technologies & Difficulty', 
 #           loc='upper left', bbox_to_anchor=(0.02, 0.98), 
 #           frameon=True, fancybox=True, shadow=True, fontsize=9,
 #           title_fontsize=11, facecolor='black', edgecolor='white', framealpha=0.9)
 # 

    graph_stats = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='lightgray',
            markersize=10, label=f'Nodes: {G.number_of_nodes()}'),
        Line2D([0], [0], linestyle='-', color='lightblue',
            linewidth=2, label=f'Edges: {edges_added}')
    ]

    plt.legend(handles=graph_stats,
            title='Connected by Technology AND Topic', 
            loc='upper left', bbox_to_anchor=(0.02, 0.98), 
            frameon=True, fancybox=True, shadow=True, fontsize=9,
            title_fontsize=11, facecolor='black', edgecolor='white', framealpha=0.9)
    
    plt.title('Personal Knowledge Graph (PKG)', 
            fontsize=20, fontweight='bold', pad=20, color='white')
    
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    return G, pos, labels_to_show, edges_added

# Run and display the network
network_result = create_PKG(True)

## Personal Knowledge Graph (PKG)


### Visual Guide:
- **Connected nodes**: Courses sharing the same technology AND topic
- **Node colors**: Blend of difficulty level and technology colors
- **Node sizes**: Based on XP reward and connection count (larger = more XP/connections)
- **Triangle nodes**: Standalone courses with unique technology-topic combinations
- **Glowing nodes**: High-value courses (>3500 XP) or highly connected courses
- **White lines**: Learning pathways between related courses

### What This Shows:
- **Course clusters**: Related learning paths in your knowledge base  
- **Knowledge gaps**: Isolated nodes reveal unique specializations
- **Learning progression**: Connected courses show natural skill development paths
- **Network density**: How interconnected your learning topics are

In [ ]:
# Skill progression spiral
def create_skill_progression_spiral():
    global merged_df
    if merged_df is None:
        print("Please load data first!")
        return

    # Sort courses by difficulty first, then by XP within each difficulty level
    df_sorted = merged_df.sort_values(['difficulty_level', 'xp']).reset_index(drop=True)

    n_courses = len(df_sorted)
    
    # Create spiral coordinates with consistent spacing
    spiral_turns = 4  # Number of complete rotations
    angle_per_course = (spiral_turns * 2 * pi) / n_courses if n_courses > 0 else 0
    
    x = []
    y = []
    
    for i, (_, course) in enumerate(df_sorted.iterrows()):
        # Calculate angle for this course
        angle = i * angle_per_course
        
        # Calculate radius based on difficulty level and position within level
        difficulty = course['difficulty_level']
        
        # Group courses by difficulty to get better radius distribution
        difficulty_courses = df_sorted[df_sorted['difficulty_level'] == difficulty]
        # position in that difficulty group (works because we reset index)
        try:
            position_in_difficulty = list(difficulty_courses.index).index(df_sorted.index[i])
        except ValueError:
            position_in_difficulty = 0
        total_in_difficulty = len(difficulty_courses)
        
        # Base radius for each difficulty level
        base_radii = {1: 1.0, 2: 2.5, 3: 4.0, 4: 5.5}
        base_radius = base_radii.get(difficulty, 1.0 + difficulty * 1.5)
        
        # Add smooth progression within each difficulty level
        if total_in_difficulty > 1:
            radius_increment = 0.8 * (position_in_difficulty / (total_in_difficulty - 1))
        else:
            radius_increment = 0
            
        radius = base_radius + radius_increment
        
        # Convert to cartesian coordinates
        x_coord = radius * np.cos(angle)
        y_coord = radius * np.sin(angle)
        
        x.append(x_coord)
        y.append(y_coord)

    # Create colors based on difficulty
    colors = []
    for _, course in df_sorted.iterrows():
        if course['difficulty_level'] <= 1:
            colors.append('#90EE90')  # Light Green
        elif course['difficulty_level'] <= 2:
            colors.append('#FFD700')  # Gold
        elif course['difficulty_level'] <= 3:
            colors.append('#FF6B6B')  # Coral Red
        else:
            colors.append('#8A2BE2')  # Blue Violet

    fig = go.Figure()

    # Add smooth spiral path
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='lines',
        line=dict(width=2, color='rgba(255,255,255,0.4)'),
        hoverinfo='skip',
        showlegend=False,
        name='Learning Path'
    ))

    # Add course markers
    max_xp = df_sorted['xp'].max() if len(df_sorted) > 0 else 0
    marker_sizes = [15 + (xp / max_xp) * 20 if max_xp > 0 else 15 for xp in df_sorted['xp']]

    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(
            size=marker_sizes,
            color=colors,
            line=dict(width=2, color='white'),
            opacity=0.9
        ),
        text=[f"<b>{row['title']}</b><br>"
              f"Technology: {row['technology_name']}<br>"
              f"Topic: {row['topic_name']}<br>"
              f"Difficulty: Level {row['difficulty_level']}<br>"
              f"XP: {row['xp']:,}<br>"
              f"Hours: {row['time_needed_in_hours']}"
              for _, row in df_sorted.iterrows()],
        hovertemplate='%{text}<br>Course #%{pointNumber}<extra></extra>',
        name='Courses',
        showlegend=False
    ))

    # Add center point
    fig.add_trace(go.Scatter(
        x=[0], y=[0],
        mode='markers+text',
        marker=dict(size=50, color='gold', symbol='star',
                    line=dict(width=4, color='white')),
        text=['START'],
        textposition="middle center",
        textfont=dict(size=13, color='black', family='Arial Black'),
        name='Journey Start',
        hoverinfo='skip',
        showlegend=False
    ))

    # Add difficulty level rings and labels
    difficulty_info = df_sorted.groupby('difficulty_level').agg({
        'xp': 'sum',
        'title': 'count'
    }).rename(columns={'title': 'count'})
    
    level_names = {1: 'Beginner', 2: 'Intermediate', 3: 'Advanced', 4: 'Expert'}
    base_radii = {1: 1.0, 2: 2.5, 3: 4.0, 4: 5.5}
    
    for difficulty in sorted(difficulty_info.index):
        ring_radius = base_radii.get(difficulty, 1.0 + difficulty * 1.5)
        
        # Add subtle ring
        theta_ring = np.linspace(0, 2*pi, 100)
        x_ring = ring_radius * np.cos(theta_ring)
        y_ring = ring_radius * np.sin(theta_ring)
        
        ring_color = '#90EE90' if difficulty == 1 else ('#FFD700' if difficulty == 2 else ('#FF6B6B' if difficulty == 3 else '#8A2BE2'))
        
        fig.add_trace(go.Scatter(
            x=x_ring, y=y_ring,
            mode='lines',
            line=dict(width=1, color=ring_color, dash='dot'),
            opacity=0.4,
            hoverinfo='skip',
            showlegend=False,
            name='Difficulty Ring'
        ))
        
        # Add level label higher above each ring
        label_x = 0
        label_y = ring_radius + 0.8 
        
        level_name = level_names.get(difficulty, f'Level {difficulty}')
        course_count = int(difficulty_info.loc[difficulty, 'count'])
        
        fig.add_trace(go.Scatter(
            x=[label_x], y=[label_y],
            mode='text',
            text=[f"<b>{level_name}</b><br>{course_count} courses"],
            textfont=dict(size=11, color=ring_color, family='Arial'),
            hoverinfo='skip',
            showlegend=False,
            name='Level Info'
        ))

    # Corner statistics (totals)
    total_xp = merged_df['xp'].sum() if 'xp' in merged_df.columns else 0
    total_videos = merged_df['num_videos'].sum() if 'num_videos' in merged_df.columns else 0
    total_exercises = merged_df['num_exercises'].sum() if 'num_exercises' in merged_df.columns else 0
    total_chapters = merged_df['num_chapters'].sum() if 'num_chapters' in merged_df.columns else 0

    corner_stats = [
        {'x': -5.2, 'y': 5.2, 'text': f'<b>Total XP</b><br>{int(total_xp):,}', 'color': '#774bf1'},
        {'x': 5.2, 'y': 5.2, 'text': f'<b>Total Videos</b><br>{int(total_videos):,}', 'color': 'lightblue'},
        {'x': -5.2, 'y': -5.2, 'text': f'<b>Total Exercises</b><br>{int(total_exercises):,}', 'color': '#bb4bbb'},
        {'x': 5.2, 'y': -5.2, 'text': f'<b>Total Chapters</b><br>{int(total_chapters):,}', 'color': 'orange'}
    ]

    for stat in corner_stats:
        fig.add_trace(go.Scatter(
            x=[stat['x']],
            y=[stat['y']],
            mode='text',
            text=[stat['text']],
            textfont=dict(size=14, color=stat['color'], family='Arial Black'),
            textposition="middle center",
            hoverinfo='skip',
            showlegend=False,
            name='Corner Stats'
        ))

    # Add legend for difficulty levels (as marker traces)
    for difficulty in sorted(difficulty_info.index):
        level_name = level_names.get(difficulty, f'Level {difficulty}')
        ring_color = '#90EE90' if difficulty == 1 else ('#FFD700' if difficulty == 2 else ('#FF6B6B' if difficulty == 3 else '#8A2BE2'))
        course_count = int(difficulty_info.loc[difficulty, 'count'])
        
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=15, color=ring_color, line=dict(width=2, color='white')),
            name=f'{level_name} ({course_count} courses)',
            showlegend=True
        ))

    fig.update_layout(
        title=dict(
            text='🌀 Learning Journey Spiral 🌀<br><sub>From Beginner Foundation to Advanced Mastery</sub>',
            x=0.5,
            font=dict(size=24, color='white')
        ),
        paper_bgcolor='rgba(10,10,30,1)',
        plot_bgcolor='rgba(10,10,30,1)',
        font=dict(color='white', size=12),
        width=1100,
        height=1000,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-7, 7]),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-7, 7], scaleanchor="x"),
        legend=dict(
            x=0.5,
            y=0.02,
            xanchor='center',
            bgcolor='rgba(0,0,0,0.8)',
            bordercolor='white',
            borderwidth=1,
            orientation='h'
        )
    )

    return fig

# Run and display spiral
spiral_fig = create_skill_progression_spiral()
if spiral_fig is not None:
    spiral_fig.show()


## Skill progression spiral

### How to Interact

**🖱️ Mouse Controls:**
- **Hover** over course dots to see detailed information (title, difficulty, XP, hours)
- **Click and drag** to pan around the spiral
- **Use zoom tools** (magnifier icons) at the top-right corner for manual zoom
- **Double-click** empty space to reset view

**📈 Visual Guide:**
- **Center star**: Starting point of your learning journey
- **Spiral path**: Shows progression from beginner to advanced courses
- **Colored dots**: Individual courses sized by XP reward
- **Dotted rings**: Difficulty level boundaries (Green→Yellow→Red→Purple)
- **Corner stats**: Total XP, videos, exercises, and chapters across all courses
- **Legend**: Course count by difficulty level

---
---
---

In [ ]:
# exporting and saving 
out_dir = Path("visualizations")
out_dir.mkdir(parents=True, exist_ok=True)


def save_plotly_html(fig, name):
    path = out_dir / f"{name}.html"
    fig.write_html(str(path))
    print(f"Saved Plotly HTML: {path.name}")

def save_matplotlib_png(fig, name):
    path = out_dir / f"{name}.png"
    try:
        fig.savefig(str(path), dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
        print(f"Saved Matplotlib PNG: {path.name}")
    except Exception as e:
        print(f"Failed to save {path.name}: {e}")
    finally:
        plt.close(fig)

# 1) Sunburst -> HTML
try:
    f = create_completed_tracks_sunburst()
    if f is not None:
        save_plotly_html(f, "01_completed_tracks_sunburst")
    else:
        print("Sunburst: nothing to save (returned None).")
except Exception as e:
    print("Sunburst save failed:", e)

# 2) Radial knowledge tree -> HTML
try:
    f = create_radial_knowledge_tree()
    if f is not None:
        save_plotly_html(f, "02_radial_knowledge_tree")
    else:
        print("Radial tree: nothing to save (returned None).")
except Exception as e:
    print("Radial tree save failed:", e)

# 3) Achievement dashboard -> HTML
try:
    f = create_achievement_dashboard()
    if f is not None:
        save_plotly_html(f, "03_achievement_dashboard")
    else:
        print("Dashboard: nothing to save (returned None).")
except Exception as e:
    print("Dashboard save failed:", e)

# 4) Skill progression timeline -> PNG (matplotlib)
try:
    fig_t = analyze_longest_track_progression()
    if fig_t is None:
        # if function plotted to current figure instead of returning, capture current fig
        fig_t = plt.gcf()
    save_matplotlib_png(fig_t, "04_longest_track_analysis")
except Exception as e:
    print("Timeline save failed:", e)


# 6) Skill progression spiral -> HTML
try:
    f = create_skill_progression_spiral()
    if f is not None:
        save_plotly_html(f, "06_skill_progression_spiral")
    else:
        print("Spiral: nothing to save (returned None).")
except Exception as e:
    print("Spiral save failed:", e)




In [ ]:
# Export the Personal Knowledge Graph (PKG)
out_dir = Path("visualizations")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "05_PKG.png"

G, pos, labels_map, edge_count = create_PKG(True)

# Error handling for empty graph
if G is None or G.number_of_nodes() == 0:
    print("No nodes in graph - skipping visualization")
else:
    # Fallback position calculation if needed
    if pos is None or len(pos) == 0:
        pos = nx.spring_layout(G, k=3, iterations=200, seed=42)

    fig = plt.figure(figsize=(16, 12), facecolor='#0a0a14')
    ax = fig.add_subplot(111)
    ax.set_facecolor('#0a0a14')

    # Draw edges
    if G.number_of_edges() > 0:
        edge_weights = [G[u][v].get('weight', 1) for u, v in G.edges()]
        max_weight = max(edge_weights) if edge_weights else 1
        edge_widths = [0.5 + (w/max_weight) * 2 for w in edge_weights]
        nx.draw_networkx_edges(G, pos, alpha=0.35, width=edge_widths, edge_color='lightblue', ax=ax)

    # Draw nodes
    node_list = list(G.nodes())
    if node_list:
        xp_values = [G.nodes[n].get('xp', 1) for n in node_list]
        max_xp = max(xp_values) if xp_values else 1

        node_colors = []
        node_sizes = []
        for n in node_list:
            nd = G.nodes[n]
            diff = nd.get('difficulty', 2)
            tech = nd.get('technology')
            base_color = difficulty_colors.get(diff, '#666666')
            tech_color = technology_colors.get(tech, '#888888')
            brgba = to_rgba(base_color)
            trgba = to_rgba(tech_color)
            blended = ((brgba[0]+trgba[0])/2, (brgba[1]+trgba[1])/2, (brgba[2]+trgba[2])/2, 0.95)
            node_colors.append(blended)
            xp = nd.get('xp', 1)
            node_sizes.append(40 + (xp / max_xp) * 120)

        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                               linewidths=1.5, edgecolors='white', ax=ax, alpha=0.9)

    # Labels
    if labels_map and pos:
        for node in node_list:
            if node in pos:
                lab = labels_map.get(node, None) if isinstance(labels_map, dict) else None
                if lab is None:
                    lab = G.nodes[node].get('title', str(node))
                x, y = pos[node]
                ax.text(x, y, lab, ha='center', va='center', fontsize=7, fontweight='bold',
                        color='white', bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.8),
                        zorder=10)

    # Legend (Nodes & Edges summary)
    graph_stats = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='lightgray',
               markersize=10, markeredgecolor='white',
               label=f'Nodes: {G.number_of_nodes()}'),
        Line2D([0], [0], linestyle='-', color='lightblue', linewidth=2,
               label=f'Edges: {edge_count}')
    ]

    ax.legend(handles=graph_stats, 
              title='Connected by Technology AND Topic',
              loc='upper left', bbox_to_anchor=(0.02, 0.98),
              frameon=True, fancybox=True, shadow=True,
              fontsize=9, title_fontsize=11,
              facecolor='black', edgecolor='white', framealpha=0.9)

    ax.set_title('Personal Knowledge Graph (PKG)', fontsize=20, color='white', pad=20)
    ax.axis('off')

    fig.savefig(str(out_path), dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    
    print(f"PKG visualization saved to {out_path}")

In [ ]:
# Zip the entire visualizations directory 

out_dir = Path("visualizations")
zip_path = out_dir.parent / "visualizations.zip"  

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in out_dir.iterdir():
        if file.is_file():
            zf.write(file, arcname=file.name)  

print(f"Created ZIP archive at: {zip_path.resolve()}")
print("Contents:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        print(" -", name)
